In [2]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import pandas as pd
from backtesting.test import SMA
import numpy as np

In [3]:
#读取文件
bu = pd.read_csv('data/bu.csv')
jd = pd.read_csv('data/jd.csv')
l = pd.read_csv('data/l.csv')
pp = pd.read_csv('data/pp.csv')
ru = pd.read_csv('data/ru.csv')
v = pd.read_csv('data/v.csv')

In [4]:
#重命名各列
def column_rename(df):
    df=df.rename(columns={
        df.columns[0]: 'Date',
        df.columns[1]: 'Open',
        df.columns[2]: 'High',
        df.columns[3]: 'Low',
        df.columns[4]: 'Close',
        df.columns[5]: 'Volume'
    }).drop(columns=[df.columns[6], df.columns[7]])
    df['Date'] = pd.to_datetime(df['Date'])
    return df.set_index('Date')
    
new_bu = column_rename(bu)
new_jd = column_rename(jd)
new_l = column_rename(l)
new_pp = column_rename(pp)
new_ru = column_rename(ru)
new_v = column_rename(v)

In [5]:
#ATR计算函数
def ATR(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = abs(df['High'] - df['Close'].shift())
    low_close = abs(df['Low'] - df['Close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()


In [6]:
# 尝试提前操作、在当日买卖（6）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma0 = self.I(SMA, price, 1)
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        self.entry_price = 0
        self.stop_loss = 0
        self.current_date = self.data.df.index[0]
        self.cross1 = self.current_date
        self.cross2 = self.current_date
        self.fall1 = self.current_date
        self.fall2 = self.current_date

    def next(self):
        current_close = self.data.Close[-1]
        self.current_date = self.data.df.index[-1]
        self.cross1 = self.current_date if crossover(self.ma0, self.ma1) else self.cross1
        self.cross2 = self.current_date if crossover(self.ma0, self.ma2) else self.cross2
        self.fall1 = self.current_date if crossover(self.ma1, self.ma0) else self.fall1
        self.fall2 = self.current_date if crossover(self.ma2, self.ma0) else self.fall2
        below_ma50 = self.data.Close[-1] < self.filter[-1]
        if self.entry_price ==0:
            buy = False
            if self.cross2==self.current_date and below_ma50 and max(self.cross1,self.fall1,self.fall2)==self.cross1 and below_ma50:
                buy = True
            if buy:
                self.buy()
                self.entry_price = current_close  # 记录入场价格
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier  # 初始止损位
        
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            if 0.95 * self.entry_price >= current_close:
                sell = True
            if current_close < self.stop_loss :
                sell = True
            elif self.fall2==self.current_date and max(self.cross1,self.fall1,self.cross2)==self.fall1 and not below_ma50:
                sell = True
            if sell:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0  # 重置止损位


In [7]:
# 添加SD计算函数
def SD(price, window=20):
    price = pd.Series(price)
    variance = price.rolling(window).var()
    sd = np.sqrt(variance)
    return sd

In [8]:
# 添加布林带指标（7）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    
    def init(self):
        # 基础均线
        self.price = self.data.Close
        self.ma1 = self.I(SMA, self.price , 10)
        self.ma2 = self.I(SMA, self.price , 20)
        self.sd2 = self.I(SD, self.price , 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        self.upper_band = self.I(lambda close: self.ma2 + self.sd2 * 2, self.data.Close)
        self.lower_band = self.I(lambda close: self.ma2 - self.sd2 * 2, self.data.Close)
        self.band_width = self.I(lambda close: self.upper_band - self.lower_band, self.data.Close)
        self.band_ma5 = self.I(SMA, self.band_width, 5)
        
        self.entry_price = 0
        self.stop_loss = 0

    def next(self):
        
        # 修改买入条件：添加持仓状态检查
        if self.entry_price ==0:
            buy = False
            if self.price[-1]>self.ma1[-1] and  self.ma1[-1] > self.ma2[-1]\
            and self.band_width[-1] > self.band_ma5[-1] and self.band_width[-1]>self.band_width[-2]:
                buy = True
            if crossover(self.price[-1], self.lower_band[-1]) :
                buy = True
            if buy:
                self.entry_price = self.price[-1]  # 记录入场价格
                self.stop_loss = self.price[-1] - self.atr[-1] * self.atr_multiplier
                self.buy()
        
        # 强化卖出条件：添加持仓状态检查
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, self.price[-1] - self.atr[-1] * self.atr_multiplier)
            if crossover(self.ma2, self.ma1) or self.price[-1] < self.stop_loss:
                sell = True
            if sell:
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0    # 重置止损位
                self.position.close()

In [9]:
# 放宽条件，让v参与交易（8）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    
    def init(self):
        # 基础均线
        self.price = self.data.Close
        self.ma1 = self.I(SMA, self.price , 10)
        self.ma2 = self.I(SMA, self.price , 20)
        self.sd2 = self.I(SD, self.price , 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        self.upper_band = self.I(lambda close: self.ma2 + self.sd2 * 2, self.data.Close)
        self.lower_band = self.I(lambda close: self.ma2 - self.sd2 * 2, self.data.Close)
        self.band_width = self.I(lambda close: self.upper_band - self.lower_band, self.data.Close)
        self.band_ma5 = self.I(SMA, self.band_width, 5)
        
        self.entry_price = 0
        self.stop_loss = 0

    def next(self):
        
        # 修改买入条件：添加持仓状态检查
        if self.entry_price ==0:
            buy = False
            if self.price[-1]>self.ma1[-1] and  self.ma1[-1] > self.ma2[-1]\
            and self.band_width[-1] > self.band_ma5[-1] and self.band_width[-1]>self.band_width[-2]\
                and self.price[-1] < self.upper_band[-1]:
                buy = True
            if crossover(self.price[-1], self.lower_band[-1]) :
                buy = True
            if self.band_width[-1] > self.band_ma5[-1] and self.band_width[-1]<self.band_width[-2]:
                buy = True
            if buy:
                self.entry_price = self.price[-1]  # 记录入场价格
                self.stop_loss = self.price[-1] - self.atr[-1] * self.atr_multiplier
                self.buy()
        
        # 强化卖出条件：添加持仓状态检查
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, self.price[-1] - self.atr[-1] * self.atr_multiplier)
            if crossover(self.ma2, self.ma1) or self.price[-1] < self.stop_loss:
                sell = True
            if sell:
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0    # 重置止损位
                self.position.close()

In [10]:
bt = Backtest(new_bu, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
print(stats)

Backtest.run:   0%|          | 0/2408 [00:00<?, ?bar/s]

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    71.54605
Equity Final [$]                   116387.962
Equity Peak [$]                    194731.976
Commissions [$]                      26862.35
Return [%]                           16.38796
Buy & Hold Return [%]                41.05305
Return (Ann.) [%]                     1.58493
Volatility (Ann.) [%]                26.22714
CAGR [%]                               1.0524
Sharpe Ratio                          0.06043
Sortino Ratio                         0.09113
Calmar Ratio                          0.03502
Alpha [%]                           -10.82817
Beta                                  0.66295
Max. Drawdown [%]                   -45.26275
Avg. Drawdown [%]                   -10.72771
Max. Drawdown Duration     1205 days 00:00:00
Avg. Drawdown Duration      150 days 00:00:00
# Trades                          

In [11]:
bt = Backtest(new_jd, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    71.99836
Equity Final [$]                     87288.88
Equity Peak [$]                      103958.4
Commissions [$]                     17984.096
Return [%]                          -12.71112
Buy & Hold Return [%]               -19.84647
Return (Ann.) [%]                    -1.39879
Volatility (Ann.) [%]                31.93872
CAGR [%]                             -0.93344
Sharpe Ratio                          -0.0438
Sortino Ratio                        -0.07759
Calmar Ratio                         -0.02489
Alpha [%]                             2.25154
Beta                                  0.75392
Max. Drawdown [%]                   -56.19388
Avg. Drawdown [%]                   -41.95001
Max. Drawdown Duration     3353 days 00:00:00
Avg. Drawdown Duration     1809 days 00:00:00
# Trades                          

In [12]:
bt = Backtest(new_l, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    70.17688
Equity Final [$]                   111531.026
Equity Peak [$]                     152204.49
Commissions [$]                     27386.504
Return [%]                           11.53103
Buy & Hold Return [%]               -18.57223
Return (Ann.) [%]                      1.1377
Volatility (Ann.) [%]                15.92194
CAGR [%]                              0.75569
Sharpe Ratio                          0.07146
Sortino Ratio                         0.10488
Calmar Ratio                          0.03877
Alpha [%]                            23.43361
Beta                                  0.64088
Max. Drawdown [%]                   -29.34438
Avg. Drawdown [%]                    -6.41324
Max. Drawdown Duration     1571 days 00:00:00
Avg. Drawdown Duration      138 days 00:00:00
# Trades                          

In [13]:
bt = Backtest(new_pp, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    74.41382
Equity Final [$]                     95462.86
Equity Peak [$]                    129753.348
Commissions [$]                     25147.576
Return [%]                           -4.53714
Buy & Hold Return [%]               -10.66985
Return (Ann.) [%]                    -0.48017
Volatility (Ann.) [%]                18.08344
CAGR [%]                              -0.3198
Sharpe Ratio                         -0.02655
Sortino Ratio                        -0.03824
Calmar Ratio                         -0.01487
Alpha [%]                             3.36417
Beta                                  0.74053
Max. Drawdown [%]                    -32.2949
Avg. Drawdown [%]                   -11.46517
Max. Drawdown Duration     3054 days 00:00:00
Avg. Drawdown Duration      400 days 00:00:00
# Trades                          

In [14]:
bt = Backtest(new_ru, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
stats

Backtest.run:   0%|          | 0/2408 [00:00<?, ?bar/s]

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    73.56086
Equity Final [$]                     132560.9
Equity Peak [$]                     182864.87
Commissions [$]                      26160.61
Return [%]                            32.5609
Buy & Hold Return [%]                10.59472
Return (Ann.) [%]                     2.96378
Volatility (Ann.) [%]                26.14525
CAGR [%]                               1.9635
Sharpe Ratio                          0.11336
Sortino Ratio                         0.19043
Calmar Ratio                          0.08233
Alpha [%]                            25.74406
Beta                                  0.64342
Max. Drawdown [%]                   -35.99915
Avg. Drawdown [%]                   -10.50937
Max. Drawdown Duration     1421 days 00:00:00
Avg. Drawdown Duration      198 days 00:00:00
# Trades                          

In [15]:
bt = Backtest(new_v, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
stats

Start                     2021-09-15 00:00:00
End                       2022-09-15 00:00:00
Duration                    365 days 00:00:00
Exposure Time [%]                    34.29752
Equity Final [$]                    85640.284
Equity Peak [$]                    112149.568
Commissions [$]                       1923.15
Return [%]                          -14.35972
Buy & Hold Return [%]                -26.5634
Return (Ann.) [%]                   -14.90654
Volatility (Ann.) [%]                15.44605
CAGR [%]                            -10.14956
Sharpe Ratio                         -0.96507
Sortino Ratio                        -1.16031
Calmar Ratio                         -0.63063
Alpha [%]                            -6.79104
Beta                                  0.28493
Max. Drawdown [%]                   -23.63744
Avg. Drawdown [%]                   -11.09676
Max. Drawdown Duration      217 days 00:00:00
Avg. Drawdown Duration      103 days 00:00:00
# Trades                          